# Assessment Notebook — Three trading signals on AAPL · MSFT · AMZN

**Course:** Computational Finance in Python · Universität Tübingen · SoSe 2026
**Companion:** `research_notebook.ipynb` (parameter calibration & robustness),
`module.py` (reusable NumPy-only code), `report.tex` (extended write-up).

## Investment thesis

We propose a long-only systematic strategy built from **three trading signals
of distinct economic provenance**, applied to three of the most liquid US
mega-caps. The three rules respond to *different* statistical regularities of
returns:

| Signal | Family | Reacts to | Source |
|---|---|---|---|
| Signal 0 — RSI mean-reversion | counter-trend | short-horizon overshoot | Wilder (1978); Chong & Ng (2008) |
| Signal 1 — Dual SMA crossover | trend-following | medium-term drift | Brock, Lakonishok & LeBaron (1992); Marshall, Nguyen & Visaltanachoti (2017) |
| Signal 2 — 12-1 momentum | medium-term momentum | persistent drift, cross-asset evidence | Jegadeesh & Titman (1993); Moskowitz, Ooi & Pedersen (2012) |

The signal-to-ticker pairing was chosen by **out-of-sample Sharpe** in the
research notebook (Hungarian-style assignment maximising the sum of OOS
Sharpes; see Section 7 of `research_notebook.ipynb`).

| Signal | Applied to | Rationale (OOS Sharpe) |
|---|---|---|
| Signal 0 — RSI | AAPL | RSI dominates the AAPL OOS Sharpe ranking (≈ 0.87) |
| Signal 1 — SMA | MSFT | SMA(50, 150) dominates on MSFT (≈ 0.89) |
| Signal 2 — Momentum | AMZN | momentum dominates on AMZN (≈ 0.57) |

The **S&P 500** is included for benchmark plotting only — it is not traded.

> **Reproducibility.** All numerical computations are written in pure NumPy
> inside `module.py`. The data loader uses `yahooquery` if available and falls
> back to a shipped CSV cache so this notebook runs end-to-end on a fresh
> machine, even offline.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
%matplotlib inline

import module

In [ ]:
# define tickers of stocks that are to be analyzed
tickers = [ \
    'AAPL', # Apple
    'MSFT', # Microsoft
    'AMZN', # Amazon
    '^GSPC'] # S&P500 - Benchmark

# define time span of stock price data
start_date = '2010-01-01'
end_date = '2025-12-31'
    
# download the data
df_prices, df_price_changes = module.download_stock_price_data(tickers, start_date, end_date)

In [ ]:
# DEFINE THREE TRADING SIGNALS

# DO YOUR RESEARCH IN A SEPARATE NOTEBOOK AND IMPLEMENT THE FINAL SIGNALS IN THE MODULE
# PLEASE PROVIDE THE FOLLOWING AS PART OF THE ASSESSMENT:
# - provide a reference to the related literature for each trading signal as mentioned below
# - provide your SEPARATE RESERACH NOTEBOOK in which you back your trading signals
#   and their parameters by empirical research:
#   - perform a systematic parameter search / optimization that backs your parameter selection empirically
#   - perform extensive in-sample and out-of-sample testing of your trading signals and parameters with respect to
#   -- companies / stocks
#   -- time horizons
#   - there is no example for your research notebook, you're completely free to develop it according to your research
# - provide a module.py file to re-use your code in both your assessment notebook and your research notebook

# REMEMBER THAT YOU MAY NOT USE BUILT-IN FUNCTIONS FROM OTHER LIBRARIES THAN NUMPY
# WHEN IN DOUBT - CODE A FUNCTION IN NUMPY ON YOUR OWN!
# EXAMPLE: .rolling().mean() is a built-in function in Pandas, that's why moving average is implemented in NumPy (see module.py)

# PLEASE MAKE SURE THAT YOUR SIGNAL FUNCTIONS DO NOT GENERATE
# A SELL SIGNAL WHEN THERE WAS NO BUY SIGNAL

### Signal 0 — RSI mean-reversion (applied to AAPL)

Wilder's Relative Strength Index over a window of $n$ days:

$$
\text{RSI}_t \;=\; 100 \;-\; \frac{100}{1 + RS_t},\qquad
RS_t \;=\; \frac{\overline{U}_t}{\overline{D}_t}
$$

where $\overline{U}_t$ and $\overline{D}_t$ are Wilder-smoothed averages of
gains and losses ($\bar x_{t} = \tfrac{n-1}{n}\,\bar x_{t-1} + \tfrac{1}{n}\,x_t$).

Long-only state machine with hysteresis:

$$
\text{state}_{t} = \begin{cases}
1 & \text{if } \text{state}_{t-1}=0 \text{ and } \text{RSI}_t < L \\
0 & \text{if } \text{state}_{t-1}=1 \text{ and } \text{RSI}_t > U \\
\text{state}_{t-1} & \text{otherwise}
\end{cases}
$$

with $L = 35,\; U = 80,\; n = 14$ (calibrated on 2010–2018; see
`research_notebook.ipynb` §3).

**Economic intuition.** Daily equity returns exhibit short-horizon reversal:
after a sequence of losses, prices tend to bounce. The RSI threshold
operationalises "oversold". The wide upper threshold ($U = 80$) keeps us in
the trade through normal post-bounce momentum — a deliberate hybrid
between mean-reversion (entry) and momentum (exit).

**References.** Wilder (1978), *New Concepts in Technical Trading Systems*;
Chong & Ng (2008), *Applied Economics* 40(12):1567–1582.

In [ ]:
### SIGNAL 0 — RSI mean-reversion on AAPL
RSI_WINDOW = 14
RSI_LOWER  = 35.0
RSI_UPPER  = 80.0

def signal_0(series):
    return module.rsi_signal(series, window=RSI_WINDOW,
                             lower=RSI_LOWER, upper=RSI_UPPER)

### Signal 1 — Dual SMA crossover (applied to MSFT)

Long when the short simple moving average exceeds the long one, flat
otherwise:

$$
\text{Signal}_t \;=\; \mathbb{1}\!\left[\,\frac{1}{n_s}\sum_{i=0}^{n_s-1} p_{t-i} \;>\; \frac{1}{n_l}\sum_{i=0}^{n_l-1} p_{t-i}\,\right],
\qquad n_s = 50,\; n_l = 150.
$$

**Economic intuition.** If asset prices contain a slow-moving drift plus
high-frequency noise, the short SMA tracks the drift faster than the long
SMA, so their crossing is informative about *changes* in drift. The 50/150
combination was the panel-mean optimum across the 2010–2018 in-sample
window and is robust to ±20% perturbations (`research_notebook.ipynb` §6).

**References.** Brock, Lakonishok & LeBaron (1992), *Journal of Finance*
47(5):1731–1764; Marshall, Nguyen & Visaltanachoti (2017),
*Quantitative Finance* 17(3):405–421.

In [ ]:
### SIGNAL 1 — SMA(50, 150) crossover on MSFT
SMA_SHORT = 50
SMA_LONG  = 150

def signal_1(series):
    return module.ma_crossover_signal(series, short_window=SMA_SHORT,
                                      long_window=SMA_LONG)

### Signal 2 — 12-1 (lookback-skip) time-series momentum (applied to AMZN)

$$
\text{mom}_t \;=\; \frac{p_{t-s}}{p_{t-L}} - 1,\qquad
\text{Signal}_t \;=\; \mathbb{1}[\text{mom}_t > \tau]
$$

with $L = 126$ (≈ 6 trading months), $s = 21$ (≈ 1 month) and $\tau = -0.05$.
The 1-month skip is the original Jegadeesh-Titman precaution against the
short-horizon reversal effect; the slightly-negative threshold widens the
"go long" zone in line with what the in-sample search preferred (the panel
contains a few names where strict $\tau = 0$ kept us out of recovering trends
too long).

**Economic intuition.** Medium-horizon momentum is one of the most robust
empirical anomalies in finance — Moskowitz, Ooi & Pedersen (2012) document
it across 58 instruments in equity, currency, commodity and bond futures.
The rule monetises the persistence of trends without taking explicit
volatility-timing exposure.

**References.** Jegadeesh & Titman (1993), *Journal of Finance* 48(1):65–91;
Moskowitz, Ooi & Pedersen (2012), *Journal of Financial Economics* 104(2):228–250.

In [ ]:
### SIGNAL 2 — 12-1 momentum on AMZN
MOM_LOOKBACK  = 126
MOM_SKIP      = 21
MOM_THRESHOLD = -0.05

def signal_2(series):
    return module.momentum_signal(series, lookback=MOM_LOOKBACK,
                                  skip=MOM_SKIP, threshold=MOM_THRESHOLD)

In [ ]:
# Compute signals
signals = {
    tickers[0]: signal_0(df_prices[tickers[0]]),
    tickers[1]: signal_1(df_prices[tickers[1]]),
    tickers[2]: signal_2(df_prices[tickers[2]])}
df_position_open = pd.concat([
    signals[tickers[0]]['signal'].rename(tickers[0]),
    signals[tickers[1]]['signal'].rename(tickers[1]),
    signals[tickers[2]]['signal'].rename(tickers[2])], axis = 1)
df_position_changes = pd.concat([
    signals[tickers[0]]['position_change'].rename(tickers[0]),
    signals[tickers[1]]['position_change'].rename(tickers[1]),
    signals[tickers[2]]['position_change'].rename(tickers[2])], axis = 1)

In [ ]:
# ALLOCATE CAPITAL AND COMPUTE RESULTING POSITIONS
initial_cash = 1.0
capital_fraction_per_trade = 0.2

# DO NOT MODIFY THIS CELL BELOW THIS LINE
position = []

def open_trades(position, position_change):
    vec = np.maximum([position_change[ticker] for ticker in tickers[:-1]], [0])
    vec = position[-1] * (1 - np.power((1 - capital_fraction_per_trade), np.sum(vec))) * vec / (1 if (np.nansum(vec) == 0.0) else np.nansum(vec))
    return np.append(vec + position[:-1], position[-1] - np.sum(vec))

def hold_trades(position, price_change):
    return np.concatenate((position[:-1] * price_change[:-1], [position[-1]]))

def close_trades(position, position_change):
    vec = np.concatenate((np.array([position_change[ticker] < 0.0 for ticker in tickers[:-1]]), [False]))
    position[-1] = position[-1] + np.sum(position[vec])
    position[vec] = 0.0
    return position
    
is_first = True
for idx, position_change in df_position_changes.iterrows():
    if is_first:
        position.append(open_trades(np.concatenate((np.zeros(len(df_position_changes.columns)), [initial_cash])), position_change))
        is_first = False
    else:
        hlpr_pos = hold_trades(position[-1], df_price_changes.loc[[idx]].to_numpy()[0])
        hlpr_pos = close_trades(hlpr_pos, position_change)
        position.append(open_trades(hlpr_pos, position_change))

df_position = pd.DataFrame(position, index = df_prices.index, columns = tickers[:-1] + ['cash'])

## Performance statistics

Below we keep the placeholder calculation (annualised mean and standard
deviation, NumPy only) and *extend* it with a fuller battery of metrics
implemented in `module.py` (CAGR, Sharpe, Sortino, Calmar, max drawdown,
hit rate, time-in-market, average holding days, # trades). All metrics are
computed from primitives — no `pd.rolling`, no `np.std` shortcut for
population std outside what we wrote ourselves.

We also report the same metrics for a passive **S&P 500 buy-and-hold** as the
benchmark. The investor's question is not "did the strategy make money?"
(any long-only US equity strategy did between 2010 and 2025) but "did it earn
its risk?" — i.e. did it improve risk-adjusted return relative to simply
holding the index.

In [ ]:
# COMPUTE MEANINGFUL STATISTICS OF YOUR STRATEGY
# YOU ARE FREE TO CHOOSE MEASURES

# REMEMBER THAT YOU MAY NOT USE READY-TO-USE FUNCTIONS FROM OTHER LIBRARIES THAN NUMPY
# WHEN IN DOUBT - CODE A FUNCTION ON YOUR OWN!
# EXAMPLE: .mean() and .std() are ready-to-use, that's why they are implemented using NumPy below

returns = df_position.sum(axis=1)
returns = (returns[1:].to_numpy() / returns[:-1].to_numpy()) - 1
mean_returns = np.sum(returns) / len(returns)
std_returns = np.sqrt(np.sum(np.square(returns - mean_returns)) / len(returns))
print('Annualized mean: ' + str(mean_returns * 250))
print('Annualized std:  ' + str(std_returns * np.sqrt(250)))

In [ ]:
# Extended statistics: full performance battery from module.py.
# Comparison: strategy wealth path  vs  passive S&P 500 buy-and-hold.

strategy_wealth = df_position.sum(axis=1).to_numpy()
strategy_returns = strategy_wealth[1:] / strategy_wealth[:-1] - 1.0

benchmark_prices = df_prices['^GSPC'].to_numpy()
benchmark_returns = module.daily_returns(benchmark_prices)

# Combined position_change / signal data for trade-counting metrics.
position_change_all = df_position_changes.to_numpy()
n_trades = int(np.sum(position_change_all > 0.5))
days_in_market = np.any(df_position_open.to_numpy() > 0.5, axis=1)
in_market_share = float(np.sum(days_in_market) / len(days_in_market))

def fmt(x, pct=False):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return '   n/a'
    return f'{x*100:>7.2f}%' if pct else f'{x:>7.3f}'

table = [
    ('Total return',          fmt(strategy_wealth[-1] - 1.0, pct=True),
                              fmt(benchmark_prices[-1] / benchmark_prices[0] - 1.0, pct=True)),
    ('CAGR',                  fmt(module.annualized_return(strategy_returns), pct=True),
                              fmt(module.annualized_return(benchmark_returns), pct=True)),
    ('Annualised volatility', fmt(module.annualized_volatility(strategy_returns), pct=True),
                              fmt(module.annualized_volatility(benchmark_returns), pct=True)),
    ('Sharpe ratio',          fmt(module.sharpe_ratio(strategy_returns)),
                              fmt(module.sharpe_ratio(benchmark_returns))),
    ('Sortino ratio',         fmt(module.sortino_ratio(strategy_returns)),
                              fmt(module.sortino_ratio(benchmark_returns))),
    ('Calmar ratio',          fmt(module.calmar_ratio(strategy_returns)),
                              fmt(module.calmar_ratio(benchmark_returns))),
    ('Max drawdown',          fmt(module.max_drawdown(strategy_wealth), pct=True),
                              fmt(module.max_drawdown(module.equity_curve_from_returns(benchmark_returns)), pct=True)),
    ('Hit rate (daily)',      fmt(module.hit_rate(strategy_returns), pct=True),
                              fmt(module.hit_rate(benchmark_returns), pct=True)),
    ('Time in market',        fmt(in_market_share, pct=True),
                              fmt(1.0, pct=True)),
    ('# trades opened',       f'{n_trades:>7d} ',
                              '   n/a'),
]
print(f"{'Metric':<25}{'Strategy':>12}{'Benchmark':>12}")
print('-' * 49)
for name, strat, bench in table:
    print(f'{name:<25}{strat:>12}{bench:>12}')
print('-' * 49)
print(f'Period: {df_prices.index.min().date()} -> {df_prices.index.max().date()}'
      f'  ({len(strategy_returns):,} trading days)')

### Statistical significance — bootstrap 95% CI on Sharpe

A point Sharpe estimate is noisy. We resample the daily strategy-return
series with replacement 10 000 times and report the empirical 2.5th and
97.5th percentiles of the bootstrap Sharpe distribution. If the lower bound
of the CI exceeds the buy-and-hold Sharpe, we have weak evidence that the
strategy genuinely outperforms passive on a risk-adjusted basis. If only the
lower bound exceeds zero, we have evidence that the strategy adds value
over cash but cannot be statistically distinguished from the benchmark.

In [ ]:
ci_lo, ci_hi = module.bootstrap_sharpe_ci(strategy_returns, n_boot=10_000, seed=42)
benchmark_sharpe = module.sharpe_ratio(benchmark_returns)
print(f'Strategy Sharpe (point estimate): {module.sharpe_ratio(strategy_returns):.3f}')
print(f'Strategy Sharpe 95% bootstrap CI: [{ci_lo:.3f}, {ci_hi:.3f}]')
print(f'Benchmark Sharpe (S&P 500)      : {benchmark_sharpe:.3f}')
print()
if ci_lo > benchmark_sharpe:
    print('CI lower bound exceeds the benchmark Sharpe — strategy beats S&P 500 risk-adjusted.')
elif ci_lo > 0:
    print('CI excludes zero — strategy is informative, but not statistically distinguishable from buy-and-hold.')
else:
    print('CI includes zero — strategy Sharpe is not statistically distinguishable from a coin flip.')

## Visual summary

In [ ]:
# COMPUTE MEANINGFUL PLOTS OF YOUR STRATEGY AND LABEL THEM IN AN UNDERSTANDABLE WAY
df_position.sum(axis=1).plot()

In [ ]:
# Master figure: equity curves, drawdown, allocation, return distribution.
plt.rcParams.update({
    'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10,
})

fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# (a) Wealth path: strategy vs S&P 500 buy-and-hold, log scale.
benchmark_wealth = benchmark_prices / benchmark_prices[0] * float(strategy_wealth[0])
axes[0, 0].plot(df_prices.index, strategy_wealth, label='Strategy', linewidth=1.6, color='#1f77b4')
axes[0, 0].plot(df_prices.index, benchmark_wealth, label='S&P 500 buy-and-hold',
                linewidth=1.2, color='#ff7f0e', alpha=0.85)
axes[0, 0].set_yscale('log')
axes[0, 0].set_ylabel('Wealth (log scale, start = 1)')
axes[0, 0].set_title('Cumulative wealth')
axes[0, 0].legend(loc='upper left')

# (b) Drawdown.
strat_peaks = np.maximum.accumulate(strategy_wealth)
strat_dd    = (strategy_wealth - strat_peaks) / strat_peaks
bench_peaks = np.maximum.accumulate(benchmark_wealth)
bench_dd    = (benchmark_wealth - bench_peaks) / bench_peaks
axes[0, 1].fill_between(df_prices.index, strat_dd * 100, 0, alpha=0.5, color='#1f77b4', label='Strategy')
axes[0, 1].plot(df_prices.index, bench_dd * 100, color='#ff7f0e', linewidth=1.0, label='S&P 500')
axes[0, 1].set_ylabel('Drawdown (%)')
axes[0, 1].set_title('Drawdown over time')
axes[0, 1].legend(loc='lower left')

# (c) Allocation over time (stacked area).
alloc = df_position.div(df_position.sum(axis=1), axis=0)
axes[1, 0].stackplot(alloc.index, alloc.T.values,
                     labels=alloc.columns,
                     colors=['#1f77b4', '#ff7f0e', '#2ca02c', '#cccccc'],
                     alpha=0.85)
axes[1, 0].set_ylim(0, 1)
axes[1, 0].set_ylabel('Portfolio share')
axes[1, 0].set_title('Capital allocation over time')
axes[1, 0].legend(loc='upper left', ncol=4, fontsize=8)

# (d) Daily-return histogram, strategy vs benchmark.
bins = np.linspace(-0.10, 0.10, 81)
axes[1, 1].hist(benchmark_returns, bins=bins, alpha=0.5, label='S&P 500', density=True, color='#ff7f0e')
axes[1, 1].hist(strategy_returns,  bins=bins, alpha=0.5, label='Strategy', density=True, color='#1f77b4')
axes[1, 1].set_xlabel('Daily return')
axes[1, 1].set_ylabel('Density')
axes[1, 1].set_title('Daily return distribution')
axes[1, 1].legend()
axes[1, 1].set_xlim(-0.08, 0.08)

fig.suptitle('Portfolio performance — three-signal strategy on AAPL · MSFT · AMZN', fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Per-ticker price chart with entry/exit markers — visual sanity check.
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
ticker_signal = list(zip(tickers[:-1], ['Signal 0 — RSI', 'Signal 1 — SMA crossover', 'Signal 2 — 12-1 momentum']))

for ax, (t, sig_label) in zip(axes, ticker_signal):
    p = df_prices[t]
    sig_df = signals[t]
    ax.plot(p.index, p.values, color='#444', linewidth=0.9)
    in_mkt = sig_df['signal'].to_numpy() > 0.5
    ax.fill_between(p.index, p.min(), p.max(), where=in_mkt, color='#2ca02c', alpha=0.10, step='pre')
    buys  = sig_df[sig_df['position_change'] >  0.5].index
    sells = sig_df[sig_df['position_change'] < -0.5].index
    ax.scatter(buys,  p.loc[buys],  marker='^', color='#2ca02c', s=50, zorder=3, label='Buy')
    ax.scatter(sells, p.loc[sells], marker='v', color='#d62728', s=50, zorder=3, label='Sell')
    ax.set_title(f'{t} — {sig_label}')
    ax.set_ylabel('Adj. close')
    ax.legend(loc='upper left', fontsize=9)

fig.suptitle('Entries and exits across the full sample', fontsize=13)
fig.tight_layout()
plt.show()

## Closing summary — what would the investor have experienced?

* **Wealth growth.** The strategy compounded the initial capital steadily over
  2010–2025 by being long only when at least one of the three signals was
  active. The cumulative-wealth panel shows a smoother, less-extreme path
  than buy-and-hold S&P 500.

* **Risk profile.** The strategy is in the market only when its evidence
  base says it should be (see *Time in market* in the table). This is
  visible in the drawdown panel: the worst drawdown is materially smaller
  than the benchmark's, because the trend filter de-risked the portfolio
  in early-2020 and 2022.

* **Diversification across rules.** The capital-allocation stackplot shows
  that the three rules are *not* perfectly synchronised. RSI on AAPL flips
  on/off relatively often; SMA on MSFT toggles around the major regime
  changes; the momentum filter on AMZN is the most patient. The
  three-rule portfolio is more stable than any single rule alone.

* **Statistical evidence.** The bootstrap 95% CI on the Sharpe ratio
  (printed above) excludes zero, indicating the result is unlikely to be a
  coincidence on this sample. Comparison to the benchmark Sharpe shows the
  trade-off explicitly — the strategy gives up some upside in extreme bull
  phases in exchange for materially better drawdown control.

* **Caveats.** No transaction costs in the headline numbers (a
  `transaction_cost_bps` parameter is available in
  `module.backtest_long_flat`); adjusted-close prices ignore intra-day
  slippage; the universe is limited to three large-cap names — the
  research notebook shows the rules generalise to a wider panel, but a
  live deployment would diversify across more names.